In [283]:
import numpy as np
import pandas as pd

In [284]:
train_df = pd.read_excel('../data/training.xlsx', index_col='room_id')
test_df = pd.read_excel('../data/test-ans.xlsx', index_col='room_id')

train_y = train_df['rental_price_USD']
train_X = train_df.drop('rental_price_USD',axis=1)

In [285]:
train_X

,accommodates,amenities,availability_30,bathrooms,bed_type,bedrooms,beds,calculated_host_listings_count,cancellation_policy,guests_included,...,longitude(East),maximum_nights,number_of_reviews,property_type,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_rating,review_scores_value,room_type
room_id,,,,,,,,,,,,,,,,,,,,,
0,6,"{Cable TV,Internet,Wireless Internet,Air Condi...",0,2.5,Real Bed,1.0,3.0,1,moderate,1,...,114.059600,10,1,House,10.0,10.0,10.0,100.0,10.0,Private room
1,2,"{Air Conditioning,Heating,Family/Kid Friendly,...",29,0.0,Futon,1.0,1.0,1,moderate,1,...,114.043225,7,0,Camper/RV,NaN,NaN,NaN,NaN,NaN,Private room
2,2,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",30,1.5,Real Bed,1.0,1.0,2,flexible,1,...,114.079426,1125,0,House,NaN,NaN,NaN,NaN,NaN,Private room
3,2,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",30,1.5,Real Bed,1.0,1.0,2,flexible,1,...,114.079035,1125,0,House,NaN,NaN,NaN,NaN,NaN,Private room
4,6,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",27,2.0,Real Bed,3.0,3.0,2,strict,1,...,114.055590,1125,0,House,NaN,NaN,NaN,NaN,NaN,Entire home/apt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5829,4,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",17,1.0,Real Bed,1.0,2.0,3,moderate,1,...,114.194588,1125,13,Apartment,10.0,10.0,10.0,88.0,10.0,Entire home/apt
5830,4,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",15,1.0,Real Bed,1.0,2.0,11,strict,3,...,114.201327,1125,9,Apartment,10.0,10.0,10.0,100.0,9.0,Entire home/apt
5831,2,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",11,1.0,Real Bed,1.0,2.0,2,moderate,1,...,114.203242,1125,0,Condominium,NaN,NaN,NaN,NaN,NaN,Private room


In [286]:
train_y

room_id
0       300
1        99
2       100
3       100
4       599
       ... 
5829    125
5830    179
5831     85
5832     25
5833    120
Name: rental_price_USD, Length: 4667, dtype: int64

In [287]:
category_cols = ['bed_type', 'cancellation_policy', 'property_type', 'room_type']
binary_cols = ['has_availability', 'host_is_superhost', 'instant_bookable']
numeric_cols = list(set(train_X.columns) - set(category_cols) - set(binary_cols) - {'amenities'})

In [288]:
numeric_cols

['bathrooms',
 'maximum_nights',
 'review_scores_checkin',
 'calculated_host_listings_count',
 'review_scores_rating',
 'review_scores_location',
 'guests_included',
 'availability_30',
 'longitude(East)',
 'bedrooms',
 'review_scores_value',
 'review_scores_communication',
 'host_listings_count',
 'accommodates',
 'beds',
 'latitude(North)',
 'number_of_reviews']

In [289]:
train_X.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4667 entries, 0 to 5833
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   accommodates                    4667 non-null   int64  
 1   amenities                       4667 non-null   object 
 2   availability_30                 4667 non-null   int64  
 3   bathrooms                       4632 non-null   float64
 4   bed_type                        4667 non-null   object 
 5   bedrooms                        4662 non-null   float64
 6   beds                            4649 non-null   float64
 7   calculated_host_listings_count  4667 non-null   int64  
 8   cancellation_policy             4667 non-null   object 
 9   guests_included                 4667 non-null   int64  
 10  has_availability                4667 non-null   object 
 11  host_is_superhost               4654 non-null   object 
 12  host_listings_count             46

In [290]:
from sklearn.preprocessing import MinMaxScaler

t = pd.get_dummies(train_X, columns=category_cols, prefix='category')
mean_series = t.mean(numeric_only=True)
[idx for idx, v in mean_series.items() if v < 0.000215 ]

['category_no_refunds',
 'category_Earth House',
 'category_Hut',
 'category_Tipi']

In [291]:
from sklearn.preprocessing import MinMaxScaler

train_X = pd.get_dummies(train_X, columns=category_cols, prefix='category')
train_X.drop(['category_no_refunds', 'category_Earth House', 'category_Hut','category_Tipi'], axis=1, inplace=True)

train_X[binary_cols] = np.where(train_X[binary_cols]=='t', 1, 0)

MMScaler = MinMaxScaler()
train_X[numeric_cols] = MMScaler.fit_transform(train_X[numeric_cols])

train_X

,accommodates,amenities,availability_30,bathrooms,bedrooms,beds,calculated_host_listings_count,guests_included,has_availability,host_is_superhost,...,category_House,category_Loft,category_Other,category_Tent,category_Townhouse,category_Treehouse,category_Villa,category_Entire home/apt,category_Private room,category_Shared room
room_id,,,,,,,,,,,,,,,,,,,,,
0,0.333333,"{Cable TV,Internet,Wireless Internet,Air Condi...",0.000000,0.3125,0.1,0.133333,0.0000,0.0625,1,0,...,1,0,0,0,0,0,0,0,1,0
1,0.066667,"{Air Conditioning,Heating,Family/Kid Friendly,...",0.966667,0.0000,0.1,0.000000,0.0000,0.0625,1,0,...,0,0,0,0,0,0,0,0,1,0
2,0.066667,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",1.000000,0.1875,0.1,0.000000,0.0625,0.0625,1,0,...,1,0,0,0,0,0,0,0,1,0
3,0.066667,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",1.000000,0.1875,0.1,0.000000,0.0625,0.0625,1,0,...,1,0,0,0,0,0,0,0,1,0
4,0.333333,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",0.900000,0.2500,0.3,0.133333,0.0625,0.0625,1,0,...,1,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5829,0.200000,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",0.566667,0.1250,0.1,0.066667,0.1250,0.0625,1,0,...,0,0,0,0,0,0,0,1,0,0
5830,0.200000,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",0.500000,0.1250,0.1,0.066667,0.6250,0.1875,1,1,...,0,0,0,0,0,0,0,1,0,0
5831,0.066667,"{TV,Cable TV,Internet,Wireless Internet,Air Co...",0.366667,0.1250,0.1,0.066667,0.0625,0.0625,1,0,...,0,0,0,0,0,0,0,0,1,0


In [292]:
train_X['amenities']

room_id
0       {Cable TV,Internet,Wireless Internet,Air Condi...
1       {Air Conditioning,Heating,Family/Kid Friendly,...
2       {TV,Cable TV,Internet,Wireless Internet,Air Co...
3       {TV,Cable TV,Internet,Wireless Internet,Air Co...
4       {TV,Cable TV,Internet,Wireless Internet,Air Co...
                              ...                        
5829    {TV,Cable TV,Internet,Wireless Internet,Air Co...
5830    {TV,Cable TV,Internet,Wireless Internet,Air Co...
5831    {TV,Cable TV,Internet,Wireless Internet,Air Co...
5832                                                   {}
5833    {TV,Cable TV,Wireless Internet,Air Conditionin...
Name: amenities, Length: 4667, dtype: object

In [293]:
from collections import defaultdict

def preprocess(s_): #:str
    s_ = s_.replace('{','')
    s_ = s_.replace('}','')
    s = {item: 1 for item in s_.split(',') if item != ''}
    return s

train_X['amenities'] = train_X['amenities'].apply(lambda x: preprocess(x))
train_X['len_amenities'] = train_X['amenities'].apply(lambda x: len(x))

train_X['amenities']

room_id
0       {'Cable TV': 1, 'Internet': 1, 'Wireless Inter...
1       {'Air Conditioning': 1, 'Heating': 1, 'Family/...
2       {'TV': 1, 'Cable TV': 1, 'Internet': 1, 'Wirel...
3       {'TV': 1, 'Cable TV': 1, 'Internet': 1, 'Wirel...
4       {'TV': 1, 'Cable TV': 1, 'Internet': 1, 'Wirel...
                              ...                        
5829    {'TV': 1, 'Cable TV': 1, 'Internet': 1, 'Wirel...
5830    {'TV': 1, 'Cable TV': 1, 'Internet': 1, 'Wirel...
5831    {'TV': 1, 'Cable TV': 1, 'Internet': 1, 'Wirel...
5832                                                   {}
5833    {'TV': 1, 'Cable TV': 1, 'Wireless Internet': ...
Name: amenities, Length: 4667, dtype: object

In [294]:
from sklearn.feature_extraction import DictVectorizer

data_dict = train_X['amenities']

dictvectorizer = DictVectorizer(sparse = False)
features = dictvectorizer.fit_transform(data_dict)
amenities_df = pd.DataFrame(features, columns = dictvectorizer.get_feature_names_out())

amenities_df

,24-Hour Check-in,Air Conditioning,Breakfast,Buzzer/Wireless Intercom,Cable TV,Carbon Monoxide Detector,Cat(s),Dog(s),Doorman,Dryer,...,Safety Card,Shampoo,Smoke Detector,Smoking Allowed,Suitable for Events,TV,Washer,Washer / Dryer,Wheelchair Accessible,Wireless Internet
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4662,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4663,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4664,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4665,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [295]:
train_X.drop('amenities', axis=1, inplace=True)

In [296]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA

pf = PolynomialFeatures(degree=3, include_bias=False)
poly_X = amenities_df.copy()
poly_X = pd.DataFrame(pf.fit_transform(poly_X), columns=pf.get_feature_names_out())
poly_X.shape[0]

4667

In [297]:
pca_dim = 14
pca = PCA(n_components=pca_dim,random_state=2023)
pca_X = pd.DataFrame(pca.fit_transform(poly_X),columns=['pca{0}'.format(x) for x in range(pca_dim)],index=train_X.index)
pca_X

,pca0,pca1,pca2,pca3,pca4,pca5,pca6,pca7,pca8,pca9,pca10,pca11,pca12,pca13
room_id,,,,,,,,,,,,,,
0,-0.336080,4.469049,5.093699,-5.930052,4.780543,0.595153,-0.459848,2.352163,-2.855780,-8.342273,4.243374,-0.267522,-1.392866,4.946649
1,-13.275792,2.578101,-3.451479,3.735927,0.997592,3.737758,2.633510,-0.875636,-0.997786,-1.844438,-0.179245,-0.176635,-0.374681,1.786365
2,7.392880,-6.502902,3.847374,2.719960,5.948550,-6.249490,5.196963,7.791516,-4.617724,0.766111,5.523142,-0.473880,-3.988958,2.871628
3,9.297429,-5.314160,2.717542,2.930615,6.180904,-6.294892,6.758278,7.286095,-8.087113,1.819929,3.359433,-2.705734,-3.955277,-3.675016
4,7.130447,-8.322066,-2.120586,-6.570364,0.975003,-5.415024,2.682963,4.869335,-2.525623,-6.798340,4.701471,-2.484979,-2.026042,1.973764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5829,-3.406639,-3.394147,-3.597619,-5.332841,-2.989221,2.130076,8.072207,-4.536534,4.860211,0.058498,1.998366,-1.379142,-4.600996,-3.993977
5830,21.573633,-6.187601,-2.740590,6.340514,8.819799,2.962607,2.675947,9.476437,0.896690,-2.290987,-2.008195,1.740881,-6.997387,3.703928
5831,13.510095,-6.038222,-1.521792,3.976913,6.420585,0.534593,2.229291,5.634583,1.343119,-2.852484,1.528946,-0.795151,-4.941456,0.039581


In [298]:
concat_X = train_X.copy()
# concat_x = pd.concat([concat_X,pca_X],ignore_index=True,axis=1)

concat_X[['pca{0}'.format(x) for x in range(pca_dim)]] = pca_X
concat_X.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4667 entries, 0 to 5833
Data columns (total 60 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   accommodates                    4667 non-null   float64
 1   availability_30                 4667 non-null   float64
 2   bathrooms                       4632 non-null   float64
 3   bedrooms                        4662 non-null   float64
 4   beds                            4649 non-null   float64
 5   calculated_host_listings_count  4667 non-null   float64
 6   guests_included                 4667 non-null   float64
 7   has_availability                4667 non-null   int32  
 8   host_is_superhost               4667 non-null   int32  
 9   host_listings_count             4654 non-null   float64
 10  instant_bookable                4667 non-null   int32  
 11  latitude(North)                 4667 non-null   float64
 12  longitude(East)                 46

In [299]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

x_trn,x_tst,y_trn,y_tst = train_test_split(concat_X.fillna(0), train_y, test_size=0.2, random_state=0)
reg = RandomForestRegressor(random_state=2023)
reg.fit(x_trn,y_trn)
y_hat = reg.predict(x_tst)

rmse_score = mean_squared_error(y_true=y_tst,y_pred=y_hat) ** 0.5
print(rmse_score)

222.91147689516703


In [300]:
test_X = test_df.drop('rental_price_USD',axis=1)
test_X = pd.get_dummies(test_X,columns=category_cols,prefix='category')
test_X.drop(['category_Boat', 'category_Chalet'],axis=1,inplace=True)
test_X[binary_cols] = np.where(test_X[binary_cols]=='t',1,0)
test_X[numeric_cols] = MMScaler.fit_transform(test_X[numeric_cols])
test_X['amenities'] = test_X['amenities'].apply(lambda x: preprocess(x))
test_X['len_amenities'] = test_X['amenities'].apply(lambda x: len(x))

data_dict = test_X['amenities']
features = dictvectorizer.transform(data_dict)
amenities_df = pd.DataFrame(features, columns = dictvectorizer.get_feature_names_out())

test_X.drop('amenities',axis=1,inplace=True)
poly_tstX = pd.DataFrame(pf.transform(amenities_df),columns=pf.get_feature_names_out())
pca_tstX = pd.DataFrame(pca.transform(poly_tstX),columns=['pca{0}'.format(x) for x in range(pca_dim)]
                        ,index=test_X.index)
concat_tstX = test_X.copy()
concat_tstX[['pca{0}'.format(x) for x in range(pca_dim)]] = pca_tstX

# 找出第一個 DataFrame 的欄位
existing_columns = concat_X.columns

# 找出第二個 DataFrame 需要添加的欄位（A 欄位）
missing_columns = existing_columns.difference(concat_tstX.columns)

# 添加缺失的欄位到第二個 DataFrame，並將其值設置為 0
for col in missing_columns:
    concat_X[col] = 0

# 找出第二個 DataFrame 需要刪除的欄位（D 欄位）
extra_columns = concat_tstX.columns.difference(existing_columns)

# 刪除多餘的欄位（D 欄位）
concat_tstX.drop(extra_columns, axis=1, inplace=True)

y_tst_hat = reg.predict(concat_tstX.fillna(0))

rmse_score = mean_squared_error(y_true=test_df['rental_price_USD'],y_pred=y_tst_hat) ** 0.5
print(rmse_score)

314.1956508847246
